# Testing & Prediction Request — Bank Marketing Deposit Model
**rahadianivan09**

Notebook ini digunakan untuk menguji endpoint prediksi model yang telah di-deploy di cloud (Hugging Face Spaces).

Model: Bank Marketing Deposit Prediction  
Endpoint: TensorFlow Serving REST API

In [1]:
# ── CELL 1: Import ───────────────────────────────
import requests
import json
import tensorflow as tf
import base64

print('Libraries loaded ✅')

2026-06-26 07:14:41.305671: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-26 07:14:42.194864: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-26 07:14:42.200331: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-26 07:14:45.969587: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Libraries loaded ✅


In [2]:
# ── CELL 2: Konfigurasi URL ──────────────────────
# Ganti dengan URL Hugging Face Space milik kamu
# Format: https://<username>-<nama-space>.hf.space (tanpa port, HF sudah arahkan ke 7860)
BASE_URL = "https://rahadianivan09-bank-deposit-prediction.hf.space"

PREDICT_URL = f"{BASE_URL}/v1/models/bank-deposit-model:predict"
STATUS_URL  = f"{BASE_URL}/v1/models/bank-deposit-model"

print(f'Predict URL: {PREDICT_URL}')

Predict URL: https://rahadianivan09-bank-deposit-prediction.hf.space/v1/models/bank-deposit-model:predict


In [3]:
# ── CELL 3: Cek Status Model ─────────────────────
response = requests.get(STATUS_URL)
print('Status Code:', response.status_code)
print('Model Status:')
print(json.dumps(response.json(), indent=2))

Status Code: 200
Model Status:
{
  "model_version_status": [
    {
      "version": "1782455796",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


In [4]:
# ── CELL 4: Sample Data ──────────────────────────
# Contoh data nasabah untuk prediksi
samples = [
    {"age": 58, "job": "management", "marital": "married", "education": "tertiary",
     "default": "no", "balance": 2143, "housing": "yes", "loan": "no",
     "contact": "unknown", "day": 5, "month": "may", "duration": 261,
     "campaign": 1, "pdays": -1, "previous": 0, "poutcome": "unknown"},
    {"age": 44, "job": "technician", "marital": "single", "education": "secondary",
     "default": "no", "balance": 29, "housing": "yes", "loan": "no",
     "contact": "unknown", "day": 5, "month": "may", "duration": 151,
     "campaign": 1, "pdays": -1, "previous": 0, "poutcome": "unknown"},
    {"age": 33, "job": "entrepreneur", "marital": "married", "education": "secondary",
     "default": "no", "balance": 2956, "housing": "yes", "loan": "no",
     "contact": "cellular", "day": 30, "month": "jul", "duration": 199,
     "campaign": 1, "pdays": -1, "previous": 0, "poutcome": "unknown"},
]

print(f'Total sample: {len(samples)} nasabah')

Total sample: 3 nasabah


In [5]:
# ── CELL 5: Fungsi Prediksi ──────────────────────
def create_example(sample):
    """Membuat tf.train.Example dari dict sample."""
    feature = {}
    int_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
    str_features = ['job', 'marital', 'education', 'default', 'housing',
                    'loan', 'contact', 'month', 'poutcome']

    for f in int_features:
        feature[f] = tf.train.Feature(int64_list=tf.train.Int64List(value=[sample[f]]))
    for f in str_features:
        feature[f] = tf.train.Feature(bytes_list=tf.train.BytesList(value=[sample[f].encode()]))

    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return base64.b64encode(example.SerializeToString()).decode()


def predict_deposit(sample):
    """Kirim request prediksi ke TF Serving endpoint."""
    payload = json.dumps({
        "signature_name": "serving_default",
        "instances": [{"examples": {"b64": create_example(sample)}}]
    })
    response = requests.post(PREDICT_URL, data=payload)
    result = response.json()
    score = result["predictions"][0][0]
    label = "Akan Deposit ✅" if score > 0.5 else "Tidak Deposit ❌"
    return score, label

print('Fungsi prediksi siap ✅')

Fungsi prediksi siap ✅


In [6]:
# ── CELL 6: Jalankan Prediksi ────────────────────
print('=' * 55)
print('   HASIL PREDIKSI — Bank Marketing Deposit Model')
print('=' * 55)

for i, sample in enumerate(samples):
    score, label = predict_deposit(sample)
    print(f"\nNasabah {i+1}")
    print(f"  Age      : {sample['age']}, Job: {sample['job']}")
    print(f"  Balance  : {sample['balance']}, Duration: {sample['duration']}s")
    print(f"  Score    : {score:.4f}")
    print(f"  Prediksi : {label}")
    print("-" * 55)

   HASIL PREDIKSI — Bank Marketing Deposit Model

Nasabah 1
  Age      : 58, Job: management
  Balance  : 2143, Duration: 261s
  Score    : 0.3168
  Prediksi : Tidak Deposit ❌
-------------------------------------------------------

Nasabah 2
  Age      : 44, Job: technician
  Balance  : 29, Duration: 151s
  Score    : 0.2996
  Prediksi : Tidak Deposit ❌
-------------------------------------------------------

Nasabah 3
  Age      : 33, Job: entrepreneur
  Balance  : 2956, Duration: 199s
  Score    : 0.4149
  Prediksi : Tidak Deposit ❌
-------------------------------------------------------
